# ЛЗ 6. Вероятностный прогноз сроков (Монте-Карло) — «Практика-Трекер»

**Исполнитель:** Сәбитов Азамат (гр. АЖ-47, Solo PM & Developer)  
**Вопрос:** *когда будет готов бэклог из N элементов?* Ответ даётся не одной датой, а распределением с перцентилями P50 / P70 / P85 / P95.

**Метод:** bootstrap по историческому throughput (число завершённых элементов за неделю): на каждом из 10 000 прогонов
недельный throughput случайно выбирается из истории (с возвращением), пока не будет закрыт весь бэклог.

In [1]:
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datetime import date, timedelta
import json

rng = np.random.default_rng(42)          # фиксируем seed: результат воспроизводим

# --- 1. Исходные данные -------------------------------------------------------
# Исторический throughput: 26 недель, элементов (issues) закрыто за неделю.
# Набор выдан преподавателем (учебный датасет курса).
history = np.array([4,6,0,7,5,2,6,9,4,5,0,6,7,4,5,1,6,5,9,3,4,6,5,8,2,5])

BACKLOG = 42            # 18 историй бэклога, декомпозированных до 42 задач-issues (INVEST)
RUNS    = 10_000
START   = date(2026, 10, 19)   # понедельник 7-й недели семестра
DEADLINE_WEEKS = 9             # недели 7..15 -> защита в конце 15-й недели (18.12.2026)

print('Недель в истории:', len(history))
print('Среднее throughput: %.2f элем./нед' % history.mean())
print('Медиана:', np.median(history), '| min:', history.min(), '| max:', history.max())
print('Недель с нулевым результатом:', int((history == 0).sum()), 'из', len(history))

Недель в истории: 26
Среднее throughput: 4.77 элем./нед
Медиана: 5.0 | min: 0 | max: 9
Недель с нулевым результатом: 2 из 26


In [2]:
# --- 2. Симуляция Монте-Карло ---------------------------------------------------
weeks = np.empty(RUNS, dtype=int)
for i in range(RUNS):
    done, w = 0, 0
    while done < BACKLOG:
        done += rng.choice(history)
        w += 1
    weeks[i] = w

def week_end(k):                      # пятница k-й недели от START
    return START + timedelta(days=(k - 1) * 7 + 4)

pcts = {p: int(np.percentile(weeks, p, method='higher')) for p in (50, 70, 85, 95)}
print('Прогонов:', RUNS, '| бэклог:', BACKLOG, 'элементов')
for p, k in pcts.items():
    print('P%d: %2d нед. -> %s' % (p, k, week_end(k).strftime('%d.%m.%Y')))
p_in_deadline = float((weeks <= DEADLINE_WEEKS).mean())
print('Вероятность уложиться в %d недель (до %s): %.1f%%' % (DEADLINE_WEEKS, week_end(DEADLINE_WEEKS).strftime('%d.%m.%Y'), p_in_deadline * 100))

Прогонов: 10000 | бэклог: 42 элементов
P50:  9 нед. -> 18.12.2026
P70: 10 нед. -> 25.12.2026
P85: 11 нед. -> 01.01.2027
P95: 12 нед. -> 08.01.2027
Вероятность уложиться в 9 недель (до 18.12.2026): 58.4%


In [3]:
# --- 3. Гистограмма и кумулятивная кривая -----------------------------------------
vals, cnt = np.unique(weeks, return_counts=True)
fig, ax = plt.subplots(figsize=(8, 4.2))
colors = ['#9fc5e8' if v <= pcts[50] else '#6fa8dc' if v <= pcts[85] else '#e06666' for v in vals]
ax.bar(vals, cnt / RUNS * 100, color=colors, edgecolor='white')
for p, c in zip((50, 85, 95), ('#38761d', '#b45f06', '#990000')):
    ax.axvline(pcts[p], color=c, ls='--', lw=1.5)
    ax.text(pcts[p] + 0.08, ax.get_ylim()[1] * 0.92, 'P%d=%d' % (p, pcts[p]), color=c, fontsize=9)
ax.axvline(DEADLINE_WEEKS + 0.5, color='black', lw=2)
ax.text(DEADLINE_WEEKS + 0.6, ax.get_ylim()[1] * 0.75, 'дедлайн\n(9 нед.)', fontsize=9)
ax.set_xlabel('Срок завершения бэклога, недель'); ax.set_ylabel('Доля прогонов, %')
ax.set_title('Распределение срока завершения (10 000 прогонов, N=%d)' % BACKLOG)
fig.tight_layout(); fig.savefig('figures/histogram.png', dpi=160); plt.close(fig)

cum = np.cumsum(cnt) / RUNS * 100
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.step(vals, cum, where='post', color='#1155cc', lw=2)
for p in (50, 70, 85, 95):
    ax.axhline(p, color='grey', ls=':', lw=0.8)
    ax.plot(pcts[p], p, 'o', color='#cc0000')
    ax.annotate('P%d: %d нед.' % (p, pcts[p]), (pcts[p], p), textcoords='offset points', xytext=(6, -12), fontsize=9)
ax.axvline(DEADLINE_WEEKS + 0.5, color='black', lw=2)
ax.set_xlabel('Срок завершения, недель'); ax.set_ylabel('Вероятность завершить к сроку, %')
ax.set_title('Кумулятивная кривая вероятности (S-кривая)')
ax.set_ylim(0, 102); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig('figures/cumulative.png', dpi=160); plt.close(fig)
print('Графики сохранены: figures/histogram.png, figures/cumulative.png')

Графики сохранены: figures/histogram.png, figures/cumulative.png


In [4]:
# --- 4. Детерминированная оценка команды (story points) vs Монте-Карло -------------
sp = {'S': 2, 'M': 5, 'L': 8}
sizes = {'S': 7, 'M': 10, 'L': 1}                       # 18 историй бэклога (docs/requirements/backlog.md)
total_sp = sum(sp[k] * v for k, v in sizes.items())     # 72 SP
VELOCITY = 8                                            # SP/нед. (оценка по ёмкости: 100 ч / ~12.5 ч на SP)
det_weeks = total_sp / VELOCITY
print('Всего SP:', total_sp, '| velocity:', VELOCITY, '| детерминированный срок: %.1f нед.' % det_weeks)

# Пересчёт после покера планирования (см. раздел ниже): 10 историй 47 SP -> 55 SP
poker_before, poker_after = 47, 55
k_adj = poker_after / poker_before
adj_sp = total_sp * k_adj
print('Коэффициент занижения по покеру: x%.2f -> скорректированный бэклог %.0f SP -> %.1f нед.' % (k_adj, adj_sp, adj_sp / VELOCITY))

mean_weeks = BACKLOG / history.mean()
print('Среднее-значение (BACKLOG / mean throughput): %.1f нед.' % mean_weeks)
print('Среднее по симуляции: %.2f нед.; медиана: %d нед.' % (weeks.mean(), pcts[50]))
print('Вероятность, что детерминированные 9 недель реально выполнимы: %.1f%%' % (float((weeks <= round(det_weeks)).mean()) * 100))

Всего SP: 72 | velocity: 8 | детерминированный срок: 9.0 нед.
Коэффициент занижения по покеру: x1.17 -> скорректированный бэклог 84 SP -> 10.5 нед.
Среднее-значение (BACKLOG / mean throughput): 8.8 нед.
Среднее по симуляции: 9.34 нед.; медиана: 9 нед.
Вероятность, что детерминированные 9 недель реально выполнимы: 58.4%


In [5]:
# --- 5. Покер планирования на 10 историях (три «шляпы»: Dev / QA / Заказчик) --------
poker = pd.DataFrame([
    # id,    разработчик, тестировщик, координатор, итог, причина расхождения
    ('US-01', 3, 5, 2, 3, 'QA: много негативных кейсов валидации формы; координатор считает форму простой'),
    ('US-02', 2, 2, 2, 2, 'Консенсус: справочник готовых баз, простой CRUD'),
    ('US-04', 3, 5, 3, 3, 'QA: граничные значения контрольного разряда БИН; алгоритм КГД известен -> 3'),
    ('US-06', 5, 5, 3, 5, 'Координатор не учитывает аудит-лог смены статусов и обязательный комментарий'),
    ('US-08', 5, 8, 3, 5, 'QA: валидация PDF и 10 МБ, риск безопасности; решили без антивируса -> 5'),
    ('US-09', 8, 5, 3, 8, 'Зависимость от шаблона юриста (EXT-01): неопределённость -> берём максимум'),
    ('US-11', 8, 8, 5, 8, 'Группировка по базам, исключённые студенты и бланк деканата'),
    ('US-12', 5, 5, 2, 5, 'Координатор занижает: роли и разграничение доступа (RBAC) нетривиальны'),
    ('US-14', 2, 3, 1, 3, 'QA: защита от перебора ИИН (rate limit) -> +1 уровень шкалы'),
    ('US-15', 13, 8, 5, 13, 'История размера L: по правилу INVEST не берётся без декомпозиции'),
], columns=['История', 'Dev', 'QA', 'Заказчик', 'Итог', 'Причина расхождения'])
poker['Разброс'] = poker[['Dev', 'QA', 'Заказчик']].max(axis=1) - poker[['Dev', 'QA', 'Заказчик']].min(axis=1)
print(poker.to_string(index=False))
print()
print('Сумма итоговых оценок:', poker['Итог'].sum(), 'SP; полный консенсус в 1 раунде: ', int((poker['Разброс'] == 0).sum()), 'из 10')
print('Средний разброс: %.1f; максимальный: %d (%s)' % (poker['Разброс'].mean(), poker['Разброс'].max(), poker.loc[poker['Разброс'].idxmax(), 'История']))

История  Dev  QA  Заказчик  Итог                                                            Причина расхождения  Разброс
  US-01    3   5         2     3 QA: много негативных кейсов валидации формы; координатор считает форму простой        3
  US-02    2   2         2     2                                Консенсус: справочник готовых баз, простой CRUD        0
  US-04    3   5         3     3    QA: граничные значения контрольного разряда БИН; алгоритм КГД известен -> 3        2
  US-06    5   5         3     5   Координатор не учитывает аудит-лог смены статусов и обязательный комментарий        2
  US-08    5   8         3     5       QA: валидация PDF и 10 МБ, риск безопасности; решили без антивируса -> 5        5
  US-09    8   5         3     8     Зависимость от шаблона юриста (EXT-01): неопределённость -> берём максимум        5
  US-11    8   8         5     8                    Группировка по базам, исключённые студенты и бланк деканата        3
  US-12    5   5         2     5

In [6]:
# --- 6. Сохранение результатов для слайда и отчёта ---------------------------------
result = {
    'runs': RUNS, 'backlog': BACKLOG, 'history_weeks': int(len(history)),
    'mean_throughput': round(float(history.mean()), 2),
    'percentiles': {str(p): {'weeks': k, 'date': week_end(k).strftime('%d.%m.%Y')} for p, k in pcts.items()},
    'p_in_deadline': round(p_in_deadline * 100, 1),
    'deadline_weeks': DEADLINE_WEEKS, 'deadline_date': week_end(DEADLINE_WEEKS).strftime('%d.%m.%Y'),
    'det_weeks': round(det_weeks, 1), 'det_adj_weeks': round(adj_sp / VELOCITY, 1),
    'total_sp': total_sp, 'adj_sp': round(adj_sp), 'velocity': VELOCITY,
    'p_det': round(float((weeks <= round(det_weeks)).mean()) * 100, 1),
    'mean_sim': round(float(weeks.mean()), 2),
    'poker_sum': int(poker['Итог'].sum()), 'poker_consensus': int((poker['Разброс'] == 0).sum()),
    'poker_mean_spread': round(float(poker['Разброс'].mean()), 1),
}
json.dump(result, open('results.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
poker.to_csv('poker_results.csv', index=False, encoding='utf-8-sig')
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "runs": 10000,
  "backlog": 42,
  "history_weeks": 26,
  "mean_throughput": 4.77,
  "percentiles": {
    "50": {
      "weeks": 9,
      "date": "18.12.2026"
    },
    "70": {
      "weeks": 10,
      "date": "25.12.2026"
    },
    "85": {
      "weeks": 11,
      "date": "01.01.2027"
    },
    "95": {
      "weeks": 12,
      "date": "08.01.2027"
    }
  },
  "p_in_deadline": 58.4,
  "deadline_weeks": 9,
  "deadline_date": "18.12.2026",
  "det_weeks": 9.0,
  "det_adj_weeks": 10.5,
  "total_sp": 72,
  "adj_sp": 84,
  "velocity": 8,
  "p_det": 58.4,
  "mean_sim": 9.34,
  "poker_sum": 55,
  "poker_consensus": 1,
  "poker_mean_spread": 3.3
}


## Выводы (в терминах вероятности)

* С вероятностью **50 %** бэклог будет закрыт за P50 недель, **85 %** — за P85 недель, **95 %** — за P95 недель (значения см. выше).
* Детерминированная оценка «72 SP / 8 SP в неделю = 9 недель» совпадает со **средним**, а значит промахивается примерно в половине случаев:
  она не учитывает «провальные» недели (throughput = 0 встречался в 2 из 26 недель) — это *flaw of averages*.
* Покер планирования показал занижение на ~17 % (47 → 55 SP на 10 историях); с поправкой детерминированная оценка сдвигается к P70–P85 симуляции.
* **Решение для заказчика:** обещать срок по **P85**, а запас до дедлайна обеспечить сокращением скоупа (US-15, US-17, US-18 → Версия 2/3).